# Work Script: g1_investigate
Converted from `sft/work/g1_investigate.py`.


In [ ]:
"""Investigate G1 mismatches: null_utterances and empty_entities_rows."""
import pandas as pd

CSV = r"c:\Users\nirmi\Desktop\Capstone\data\processed\integrated_conversations_ner_validated.csv"

df = pd.read_csv(
    CSV,
    dtype={"dialogue_id": "str", "original_id": "str"},
    low_memory=False,
    encoding="utf-8",
)

print("=== NULL UTTERANCES ===")
null_mask = df["utterance"].isna()
print(f"  pd.isna count              : {null_mask.sum()}")
# Check for strings that look null
empty_str = (df["utterance"].astype(str).str.strip() == "")
print(f"  empty string after strip   : {empty_str.sum()}")
nan_str   = (df["utterance"].astype(str).str.lower() == "nan")
print(f"  string 'nan'               : {nan_str.sum()}")
combined  = null_mask | empty_str | nan_str
print(f"  null | empty | 'nan' string: {combined.sum()}")

print()
print("=== EMPTY ENTITIES ROWS ===")
# DATASET.md: 46,889 rows with empty entities (entity_count == 0)
ec0_mask = (df["entity_count"].fillna(0) == 0)
print(f"  entity_count == 0 (or NaN) : {ec0_mask.sum()}")

# Alternative: check the string columns
ents_empty = df["extracted_entities"].fillna("").astype(str).str.strip() == ""
print(f"  extracted_entities empty   : {ents_empty.sum()}")

# Combined: both empty
both_empty = ec0_mask & ents_empty
print(f"  both ec0 AND ents empty    : {both_empty.sum()}")

# DATASET.md says "entity_count == 0, both columns are empty strings (not NaN)"
# Let's check if some ec=0 rows have non-empty extracted_entities
ec0_df = df[ec0_mask]
ec0_ents_nonempty = ec0_df["extracted_entities"].fillna("").astype(str).str.strip() != ""
print(f"  ec0 rows with non-empty extracted_entities: {ec0_ents_nonempty.sum()}")

# Original DATASET.md definition: "rows with empty entities"
# The doc says entity_count == 0 → both columns empty strings
# measured = 48,768, expected = 46,889 → diff = 1,879
print(f"\n  Difference: {48768 - 46889} rows")

# Check entity_count distribution
print("\n  entity_count value counts (top):")
print(df["entity_count"].value_counts().head(5).to_string())

In [ ]:
print("RESULT: PASS g1_investigate.py notebook compiled successfully")